# 06b-h — forensic congelato della generalizzazione del voltaggio

Nessuna rete viene allenata. Quattro famiglie di checkpoint già congelate vengono confrontate con correzioni preregistrate di ampiezza e bias. Il settimo componente train serve alla calibrazione; l'ottavo è un audit sigillato.

In [ ]:
import importlib,json,os,shutil,subprocess,sys,time,zipfile
from pathlib import Path
ELM_REPOSITORY='https://github.com/Zagred47/giada.git';ELM_REF=os.environ.get('HAYFLOW_ELM_REF','main');ROOT=Path('/kaggle/working');ELM_REPO=ROOT/'hayflow_workspace'/'elmneuron';ELM_REPO.parent.mkdir(parents=True,exist_ok=True)
def run(command,cwd=None):print('+',' '.join(map(str,command)),flush=True);subprocess.run(list(map(str,command)),cwd=cwd,check=True)
if not (ELM_REPO/'.git').is_dir():run(['git','clone',ELM_REPOSITORY,ELM_REPO])
run(['git','fetch','origin',ELM_REF],cwd=ELM_REPO);run(['git','checkout','--detach','FETCH_HEAD'],cwd=ELM_REPO);REVISION=subprocess.check_output(['git','rev-parse','HEAD'],cwd=ELM_REPO,text=True).strip();sys.path.insert(0,str(ELM_REPO));importlib.invalidate_caches();print({'revision':REVISION})

## 1. Sorgenti immutabili

Sono richiesti il dataset composito, gli artefatti della catena 05t–06b-f e il nuovo risultato 06b-g. Gli artefatti decisionali vengono individuati tramite il loro indice SHA-256 anche quando Kaggle li monta come `archive.zip`.

In [ ]:
from src.hayflow_model.rollout_aware_architecture_canary import discover_indexed_artifact_source
from src.hayflow_model.atomic_state_dynamics_playground import EXPECTED_05T_INDEX_SHA256
from src.hayflow_model.causal_voltage_state_coupling_forensic import EXPECTED_06B_INDEX_SHA256
from src.hayflow_model.causal_voltage_bridge_representation_forensic import EXPECTED_06BB_INDEX_SHA256
from src.hayflow_model.nested_coupling_optimization_scaling_forensic import EXPECTED_06BC_INDEX_SHA256
from src.hayflow_model.recursive_voltage_state_contract_forensic import EXPECTED_06BD_INDEX_SHA256
from src.hayflow_model.recursive_joint_repair_matrix import EXPECTED_06BE_INDEX_SHA256
from src.hayflow_model.state_scheduled_sampling_confirmation import EXPECTED_06BF_INDEX_SHA256
from src.hayflow_model.frozen_voltage_generalization_forensic import EXPECTED_06BG_INDEX_SHA256
INPUT_ROOT=Path('/kaggle/input')
def indexed(name,expected,env):
 override=os.environ.get(env);source=discover_indexed_artifact_source(INPUT_ROOT,expected,override=Path(override) if override else None);assert source is not None,f'Artefatto {name} esatto non trovato.';return source
ARTIFACT_05T_SOURCE=indexed('05t',EXPECTED_05T_INDEX_SHA256,'HAYFLOW_05T_ARTIFACT');ARTIFACT_06B_SOURCE=indexed('06b',EXPECTED_06B_INDEX_SHA256,'HAYFLOW_06B_ARTIFACT');ARTIFACT_06BB_SOURCE=indexed('06b-b',EXPECTED_06BB_INDEX_SHA256,'HAYFLOW_06BB_ARTIFACT');ARTIFACT_06BC_SOURCE=indexed('06b-c',EXPECTED_06BC_INDEX_SHA256,'HAYFLOW_06BC_ARTIFACT');ARTIFACT_06BD_SOURCE=indexed('06b-d',EXPECTED_06BD_INDEX_SHA256,'HAYFLOW_06BD_ARTIFACT')
override_06be=os.environ.get('HAYFLOW_06BE_ARTIFACT');ARTIFACT_06BE_SOURCE=None
for expected in EXPECTED_06BE_INDEX_SHA256:
 candidate=discover_indexed_artifact_source(INPUT_ROOT,expected,override=Path(override_06be) if override_06be else None)
 if candidate is not None:ARTIFACT_06BE_SOURCE=candidate;break
assert ARTIFACT_06BE_SOURCE is not None,'Artefatto 06b-e canonico o confermativo non trovato.'
ARTIFACT_06BF_SOURCE=indexed('06b-f',EXPECTED_06BF_INDEX_SHA256,'HAYFLOW_06BF_ARTIFACT');ARTIFACT_06BG_SOURCE=indexed('06b-g',EXPECTED_06BG_INDEX_SHA256,'HAYFLOW_06BG_ARTIFACT')
def extract_zip_safely(source,destination):
 source,destination=Path(source),Path(destination);marker=destination/'.source_stamp';stamp=f'{source.stat().st_size}:{source.stat().st_mtime_ns}'
 if marker.is_file() and marker.read_text().strip()==stamp:return destination
 if destination.exists():shutil.rmtree(destination)
 destination.mkdir(parents=True);root=destination.resolve()
 with zipfile.ZipFile(source) as archive:
  for member in archive.infolist():
   target=(destination/member.filename).resolve();assert target==root or root in target.parents,member.filename
  archive.extractall(destination)
 marker.write_text(stamp);return destination
topup_override=os.environ.get('HAYFLOW_TOPUP_V3');topup_candidates=([Path(topup_override).expanduser()] if topup_override else [])+list(INPUT_ROOT.rglob('hayflow_bap_validation_support_topup_v3.zip'))+[p.parent for p in INPUT_ROOT.rglob('composite_dataset_manifest.json')]
TOPUP_SOURCE=next((p.resolve() for p in topup_candidates if p.exists()),None);assert TOPUP_SOURCE is not None,'Top-up BAP v3 non trovato.';TOPUP_ROOT=extract_zip_safely(TOPUP_SOURCE,'/kaggle/working/hayflow06bh_topup') if TOPUP_SOURCE.is_file() else TOPUP_SOURCE
manifest_candidates=list(Path(TOPUP_ROOT).rglob('composite_dataset_manifest.json'));assert len(manifest_candidates)==1,manifest_candidates;COMPOSITE_MANIFEST=manifest_candidates[0]
base_override=os.environ.get('HAYFLOW_BASE_DATASET');base_candidates=([Path(base_override).expanduser()] if base_override else [])+[p.parent for p in INPUT_ROOT.rglob('transition_dataset.h5') if 'targeted' in str(p).lower() and 'topup' not in str(p).lower()]+[p for p in INPUT_ROOT.rglob('archive.zip') if 'hayflow-targeted-transition-dataset' in str(p).lower()]
BASE_SOURCE=next((p.resolve() for p in base_candidates if p.exists()),None);assert BASE_SOURCE is not None,'Dataset base targeted v1.1 non trovato.'
print({'05t':str(ARTIFACT_05T_SOURCE),'06b':str(ARTIFACT_06B_SOURCE),'06b-b':str(ARTIFACT_06BB_SOURCE),'06b-c':str(ARTIFACT_06BC_SOURCE),'06b-d':str(ARTIFACT_06BD_SOURCE),'06b-e':str(ARTIFACT_06BE_SOURCE),'06b-f':str(ARTIFACT_06BF_SOURCE),'06b-g':str(ARTIFACT_06BG_SOURCE),'base':str(BASE_SOURCE)})

In [ ]:
from src.hayflow_data import prepare_composite_flowmap_bundle
hash_started={};hash_last={}
def hash_progress(name,done,total):
 now=time.monotonic();hash_started.setdefault(name,now);percent=int(100*done/total)
 if percent>=hash_last.get(name,-10)+10 or done==total:
  elapsed=now-hash_started[name];rate=done/max(elapsed,1e-9);eta=(total-done)/max(rate,1e-9);print(f'[HayFlow 06b-h][SHA-256 {name}] {percent}% ETA {eta/60:.1f} min',flush=True);hash_last[name]=percent
bundle=prepare_composite_flowmap_bundle(COMPOSITE_MANIFEST,base_source=BASE_SOURCE,progress=hash_progress);assert bundle.manifest['valid'] and bundle.transition_count==29880;print({'dataset_valid':True,'transitions':bundle.transition_count,'fingerprint':bundle.fingerprint})

## 2. Preflight e ruoli sigillati

Il preflight verifica ogni membro del 06b-g, congela tutti i parametri e costruisce calibrazione e audit dai due componenti train successivi. Nessun output di audit entra nella selezione.

In [ ]:
import yaml
from IPython.display import display
from src.hayflow_model import FrozenVoltageForensicConfig,FrozenVoltageGeneralizationForensic
base_cfg=yaml.safe_load((ELM_REPO/'configs/hayflow/hayflow_recursive_joint_repair_matrix.yml').read_text())['recursive_joint_repair_matrix'];scheduled=yaml.safe_load((ELM_REPO/'configs/hayflow/hayflow_state_scheduled_sampling_confirmation.yml').read_text())['state_scheduled_sampling_confirmation'];forensic=yaml.safe_load((ELM_REPO/'configs/hayflow/hayflow_frozen_voltage_generalization_forensic.yml').read_text())['frozen_voltage_generalization_forensic'];base_cfg.update(scheduled);base_cfg.update(forensic);config=FrozenVoltageForensicConfig.from_mapping(base_cfg)
OUTPUT_DIR=Path('/kaggle/working/artifacts/hayflow_frozen_voltage_generalization_forensic');assert not OUTPUT_DIR.exists(),f'Output gia presente: {OUTPUT_DIR}. Avvia una sessione nuova.'
session=FrozenVoltageGeneralizationForensic(bundle,OUTPUT_DIR,config,ARTIFACT_05T_SOURCE,ARTIFACT_06B_SOURCE,ARTIFACT_06BB_SOURCE,ARTIFACT_06BC_SOURCE,ARTIFACT_06BD_SOURCE,ARTIFACT_06BE_SOURCE,ARTIFACT_06BF_SOURCE,ARTIFACT_06BG_SOURCE,code_revision=REVISION);contract=session.prepare_frozen_voltage_forensic();roles=contract['forensic_roles']
display({'valid':contract['valid'],'06b-g':contract['source_06bg'],'frozen_arms':contract['frozen_model_arms'],'calibration_trajectories':roles['trajectory_counts']['voltage_calibration'],'audit_trajectories':roles['trajectory_counts']['voltage_audit'],'component_positions':roles['role_component_positions'],'overlaps':roles['overlaps'],'neural_training':contract['neural_training_performed']});assert contract['valid'] and not any(roles['overlaps'].values()) and not contract['neural_training_performed']

## 3. Calibrazione congelata e audit

La calibrazione confronta 40 candidati senza aggiornare pesi neurali. Due tracker compatti mostrano avanzamento ed ETA del prefit del bias e della matrice. Una sola configurazione viene poi valutata sull'audit, con metriche separate per intensità della transizione.

In [ ]:
calibration_report=session.calibrate_frozen_voltage();selected=calibration_report['selected']
display({'valid':calibration_report['valid'],'candidate_count':calibration_report['candidate_count'],'eligible':calibration_report['eligible_candidate_count'],'selected':{key:selected[key] for key in ('model_arm','alpha','bias_mode','bias_mv_per_ms','median_voltage_error_ratio','median_state_gain')},'audit_accessed':calibration_report['audit_accessed']});assert calibration_report['valid'] and calibration_report['candidate_count']==40 and not calibration_report['audit_accessed']

In [ ]:
audit_report=session.audit_frozen_voltage(calibration_report);final_report=session.finalize_frozen_voltage_forensic(calibration_report,audit_report)
display({'valid':final_report['valid'],'diagnosis':final_report['diagnosis'],'selected':final_report['selected_frozen_candidate'],'audit_medians':final_report['audit_medians'],'gates':final_report['gate_checks'],'06c_authorized':final_report['coupled_06c_canary_authorized'],'next_step':final_report['next_step']});assert final_report['valid'] and not final_report['coupled_06c_canary_authorized'] and not final_report['full_training_authorized']

## 4. Download stabile

La cella crea lo ZIP in `/kaggle/working`, lo converte in base64 e avvia il download tramite un Blob nel browser.

In [ ]:
from shutil import make_archive
import base64
from IPython.display import Javascript,display
zip_path=Path(make_archive('/kaggle/working/hayflow_frozen_voltage_generalization_forensic','zip',root_dir=OUTPUT_DIR.parent,base_dir=OUTPUT_DIR.name));payload=base64.b64encode(zip_path.read_bytes()).decode('ascii');filename=zip_path.name
display(Javascript(f"""const binary=atob('{payload}');const bytes=new Uint8Array(binary.length);for(let i=0;i<binary.length;i++)bytes[i]=binary.charCodeAt(i);const blob=new Blob([bytes],{{type:'application/zip'}});const url=URL.createObjectURL(blob);const a=document.createElement('a');a.href=url;a.download='{filename}';document.body.appendChild(a);a.click();a.remove();setTimeout(()=>URL.revokeObjectURL(url),60000);"""));print({'zip':str(zip_path),'size_mib':round(zip_path.stat().st_size/2**20,2),'download':'avviato dal browser'})